# Diabetes Linear Model 代码附录

## `scripts/run_analysis.py`

In [ ]:
"""Run the Hoff (2009) azdiabetes linear model analysis."""
from __future__ import annotations

import argparse
import random
import sys
from pathlib import Path


def _bootstrap_sys_path() -> None:
    """Add the repository's ``src`` directory to ``sys.path`` if present.

    The script may be placed either inside ``scripts/`` (as in this repository)
    or copied to the project root. Rather than assuming a fixed relative
    location, walk up the directory tree until a folder containing ``src`` is
    found and prepend it to ``sys.path``.
    """

    current = Path(__file__).resolve().parent
    for parent in (current, *current.parents):
        candidate = parent / "src"
        if candidate.exists():
            path_str = str(candidate)
            if path_str not in sys.path:
                sys.path.insert(0, path_str)
            break
    else:  # pragma: no cover - defensive fallback, should not happen in repo
        raise RuntimeError(
            "Unable to locate the 'src' directory. Ensure the script resides "
            "within the Diabetes-Linear-Model project structure."
        )


_bootstrap_sys_path()

from diabetes_linear_model.data import load_az_diabetes, prepare_regression_matrices
from diabetes_linear_model.gprior import fit_g_prior_model
from diabetes_linear_model.model_selection import GPriorModelSelector


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--data-path",
        type=Path,
        default=Path("data/azdiabetes.dat.txt"),
        help="Path to the azdiabetes data file.",
    )
    parser.add_argument("--response", default="glu", help="Name of the response variable")
    parser.add_argument(
        "--exclude",
        nargs="*",
        default=["diabetes"],
        help="Columns to exclude from the predictor matrix.",
    )
    parser.add_argument("--g", type=float, default=None, help="Value of Zellner's g. Defaults to the sample size.")
    parser.add_argument("--nu0", type=float, default=2.0, help="Prior degrees of freedom for sigma^2")
    parser.add_argument("--sigma0-sq", dest="sigma0_sq", type=float, default=1.0, help="Prior scale for sigma^2")
    parser.add_argument("--seed", type=int, default=2024, help="Random seed")
    parser.add_argument("--samples", type=int, default=5000, help="Posterior samples for part (a)")
    parser.add_argument("--selection-samples", type=int, default=4000, help="Number of retained Gibbs samples for part (b)")
    parser.add_argument("--burn-in", type=int, default=1000, help="Burn-in iterations for the Gibbs sampler")
    parser.add_argument("--thin", type=int, default=5, help="Thinning interval for the Gibbs sampler")
    return parser.parse_args()


def print_coefficient_table(names: list[str], means: list[float], lowers: list[float], uppers: list[float]) -> None:
    width = max(len(name) for name in names) + 2
    header = f"{'variable'.ljust(width)}{'mean':>12}{'lower':>12}{'upper':>12}"
    print(header)
    print("-" * len(header))
    for name, mean, lower, upper in zip(names, means, lowers, uppers):
        print(f"{name.ljust(width)}{mean:12.3f}{lower:12.3f}{upper:12.3f}")


def main() -> None:
    args = parse_args()
    rng = random.Random(args.seed)

    data = load_az_diabetes(args.data_path)
    predictors, _, X, y = prepare_regression_matrices(data, response=args.response, exclude=args.exclude)

    g_value = args.g if args.g is not None else float(len(X))

    print("=== Part (a): Full model with g-prior ===")
    posterior = fit_g_prior_model(
        X,
        y,
        g=g_value,
        nu0=args.nu0,
        sigma0_sq=args.sigma0_sq,
        num_samples=args.samples,
        rng=rng,
    )
    print_coefficient_table(
        predictors,
        posterior.beta_mean,
        posterior.beta_ci[0],
        posterior.beta_ci[1],
    )
    print(
        "Sigma^2 mean: {:.3f}, 95% CI: ({:.3f}, {:.3f})".format(
            posterior.sigma2_mean, posterior.sigma2_ci[0], posterior.sigma2_ci[1]
        )
    )

    print("\n=== Part (b): Model selection and averaging ===")
    selector = GPriorModelSelector(
        X,
        y,
        g=g_value,
        nu0=args.nu0,
        sigma0_sq=args.sigma0_sq,
        rng=rng,
    )
    result = selector.run(
        num_samples=args.selection_samples,
        burn_in=args.burn_in,
        thin=args.thin,
    )

    print("Posterior inclusion probabilities (excluding intercept):")
    for name, prob in zip(predictors[1:], result.inclusion_probabilities):
        print(f"  {name}: {prob:.3f}")

    print("\nPosterior coefficient summaries (model averaged):")
    print_coefficient_table(
        predictors,
        result.beta_mean,
        result.beta_ci[0],
        result.beta_ci[1],
    )
    print(
        "Sigma^2 mean: {:.3f}, 95% CI: ({:.3f}, {:.3f})".format(
            result.sigma2_mean, result.sigma2_ci[0], result.sigma2_ci[1]
        )
    )


if __name__ == "__main__":
    main()


## `src/diabetes_linear_model/__init__.py`

In [ ]:
"""Utilities for Bayesian linear modeling of the azdiabetes dataset."""

from .data import TabularData, load_az_diabetes, prepare_regression_matrices
from .gprior import GPriorPosterior, fit_g_prior_model
from .model_selection import GPriorModelSelector, ModelSelectionResult

__all__ = [
    "TabularData",
    "load_az_diabetes",
    "prepare_regression_matrices",
    "GPriorPosterior",
    "fit_g_prior_model",
    "GPriorModelSelector",
    "ModelSelectionResult",
]


## `src/diabetes_linear_model/data.py`

In [ ]:
"""Data loading utilities for the azdiabetes dataset."""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List, Sequence


@dataclass
class TabularData:
    columns: List[str]
    rows: List[List[str]]


def _split_row(row: str) -> List[str]:
    return row.strip().split()


def _manual_parse(lines: Iterable[str]) -> TabularData:
    iterator = iter(lines)
    header = next(iterator, None)
    if header is None:
        raise ValueError("The provided file is empty.")
    columns = _split_row(header)
    rows = [_split_row(line) for line in iterator if line.strip()]
    if any(len(row) != len(columns) for row in rows):
        raise ValueError("Inconsistent number of columns in the data file.")
    return TabularData(columns=columns, rows=rows)


def load_az_diabetes(path: str | Path, categorical: Sequence[str] = ("diabetes",)) -> TabularData:
    """Load the azdiabetes dataset from a whitespace separated file."""

    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    with path.open("r", encoding="utf-8") as handle:
        data = _manual_parse(handle.readlines())

    # Coerce numeric columns where appropriate.
    cat_set = set(categorical)
    for row in data.rows:
        for idx, column in enumerate(data.columns):
            if column in cat_set:
                continue
            row[idx] = float(row[idx])
    return data


def prepare_regression_matrices(
    data: TabularData,
    response: str,
    exclude: Sequence[str] | None = None,
    add_intercept: bool = True,
) -> tuple[List[str], str, List[List[float]], List[float]]:
    """Construct a design matrix and response vector from tabular data."""

    if response not in data.columns:
        raise KeyError(f"Response column '{response}' is not present in the data.")

    exclude = set(exclude or []) | {response}

    predictor_columns = [col for col in data.columns if col not in exclude]
    if add_intercept:
        predictors = ["intercept"] + predictor_columns
    else:
        predictors = predictor_columns.copy()

    response_index = data.columns.index(response)
    predictor_indices = [data.columns.index(col) for col in predictor_columns]

    X: List[List[float]] = []
    y: List[float] = []
    for row in data.rows:
        y.append(float(row[response_index]))
        features = [float(row[idx]) for idx in predictor_indices]
        if add_intercept:
            X.append([1.0] + features)
        else:
            X.append(features)

    return predictors, response, X, y


## `src/diabetes_linear_model/gprior.py`

In [ ]:
"""Posterior computations for g-prior linear regression."""
from __future__ import annotations

import math
import random
from dataclasses import dataclass
from typing import List, Tuple

from .linalg import (
    Matrix,
    Vector,
    add_vectors,
    cholesky,
    matvec,
    matmul,
    quantiles,
    scale_vector,
    solve_pos_def,
    transpose,
    vecdot,
    vector_mean,
    vector_quantiles,
)


@dataclass
class GPriorPosterior:
    """Summary of the posterior distribution under a g-prior."""

    beta_mean: Vector
    beta_ci: List[Vector]
    sigma2_mean: float
    sigma2_ci: Tuple[float, float]
    beta_samples: List[Vector]
    sigma2_samples: List[float]


def _posterior_hyperparameters(
    X: Matrix, y: Vector, g: float
) -> Tuple[Vector, Matrix, Matrix, float]:
    Xt = transpose(X)
    XtX = matmul(Xt, X)
    Xty = matvec(Xt, y)
    beta_ols = solve_pos_def(XtX, Xty)
    shrinkage = g / (g + 1.0)
    beta_mean = scale_vector(beta_ols, shrinkage)
    ssr_g = vecdot(y, y) - shrinkage * vecdot(Xty, beta_ols)
    chol = cholesky(XtX)
    return beta_mean, XtX, chol, ssr_g


def _sample_beta(
    chol: Matrix,
    beta_mean: Vector,
    sigma2: float,
    g: float,
    rng: random.Random,
) -> Vector:
    shrinkage = g / (g + 1.0)
    std_normals = [rng.gauss(0.0, 1.0) for _ in beta_mean]
    Lt = transpose(chol)
    v = [0.0] * len(beta_mean)
    for i in range(len(beta_mean) - 1, -1, -1):
        s = sum(Lt[i][k] * v[k] for k in range(i + 1, len(beta_mean)))
        v[i] = (std_normals[i] - s) / Lt[i][i]
    scale = math.sqrt(shrinkage * sigma2)
    return add_vectors(beta_mean, [scale * value for value in v])


def fit_g_prior_model(
    X: Matrix,
    y: Vector,
    g: float,
    nu0: float,
    sigma0_sq: float,
    num_samples: int = 5000,
    cred_mass: float = 0.95,
    rng: random.Random | None = None,
) -> GPriorPosterior:
    """Draw posterior samples for a g-prior regression model."""

    if len(X) != len(y):
        raise ValueError("X and y must have the same number of observations.")
    if not X or not X[0]:
        raise ValueError("Design matrix cannot be empty.")

    rng = rng or random.Random()

    beta_mean_conditional, XtX, chol, ssr_g = _posterior_hyperparameters(X, y, g)

    shape = 0.5 * (nu0 + len(X))
    scale = 0.5 * (nu0 * sigma0_sq + ssr_g)

    sigma2_samples: List[float] = []
    beta_samples: List[Vector] = []

    for _ in range(num_samples):
        sigma2 = 1.0 / rng.gammavariate(shape, 1.0 / scale)
        sigma2_samples.append(sigma2)
        beta_samples.append(_sample_beta(chol, beta_mean_conditional, sigma2, g, rng))

    lower = (1.0 - cred_mass) / 2.0
    upper = 1.0 - lower
    beta_ci = quantiles(beta_samples, [lower, upper])
    sigma2_ci = tuple(vector_quantiles(sigma2_samples, [lower, upper]))

    return GPriorPosterior(
        beta_mean=vector_mean(beta_samples),
        beta_ci=beta_ci,
        sigma2_mean=sum(sigma2_samples) / len(sigma2_samples),
        sigma2_ci=sigma2_ci,
        beta_samples=beta_samples,
        sigma2_samples=sigma2_samples,
    )


def analytic_posterior_mean(X: Matrix, y: Vector, g: float) -> Vector:
    Xt = transpose(X)
    XtX = matmul(Xt, X)
    Xty = matvec(Xt, y)
    beta_ols = solve_pos_def(XtX, Xty)
    shrinkage = g / (g + 1.0)
    return scale_vector(beta_ols, shrinkage)


def analytic_ssr_g(X: Matrix, y: Vector, g: float) -> float:
    Xt = transpose(X)
    XtX = matmul(Xt, X)
    Xty = matvec(Xt, y)
    beta_ols = solve_pos_def(XtX, Xty)
    shrinkage = g / (g + 1.0)
    return vecdot(y, y) - shrinkage * vecdot(Xty, beta_ols)


## `src/diabetes_linear_model/linalg.py`

In [ ]:
"""Lightweight linear algebra utilities using pure Python."""
from __future__ import annotations

import math
from typing import Iterable, List

Vector = List[float]
Matrix = List[List[float]]


def zeros(rows: int, cols: int) -> Matrix:
    return [[0.0 for _ in range(cols)] for _ in range(rows)]


def identity(n: int) -> Matrix:
    mat = zeros(n, n)
    for i in range(n):
        mat[i][i] = 1.0
    return mat


def transpose(matrix: Matrix) -> Matrix:
    if not matrix:
        return []
    return [list(row) for row in zip(*matrix)]


def matmul(A: Matrix, B: Matrix) -> Matrix:
    result = zeros(len(A), len(B[0]))
    for i, row in enumerate(A):
        for k, a in enumerate(row):
            if a == 0.0:
                continue
            for j, b in enumerate(B[k]):
                result[i][j] += a * b
    return result


def matvec(A: Matrix, v: Vector) -> Vector:
    return [sum(a * b for a, b in zip(row, v)) for row in A]


def vecdot(a: Vector, b: Vector) -> float:
    return sum(x * y for x, y in zip(a, b))


def cholesky(A: Matrix) -> Matrix:
    n = len(A)
    L = zeros(n, n)
    for i in range(n):
        for j in range(i + 1):
            s = sum(L[i][k] * L[j][k] for k in range(j))
            if i == j:
                value = A[i][i] - s
                if value <= 0:
                    raise ValueError("Matrix is not positive definite.")
                L[i][j] = math.sqrt(value)
            else:
                if L[j][j] == 0:
                    raise ValueError("Matrix is singular.")
                L[i][j] = (A[i][j] - s) / L[j][j]
    return L


def solve_lower_triangular(L: Matrix, b: Vector) -> Vector:
    y = [0.0] * len(L)
    for i in range(len(L)):
        s = sum(L[i][k] * y[k] for k in range(i))
        y[i] = (b[i] - s) / L[i][i]
    return y


def solve_upper_triangular(U: Matrix, b: Vector) -> Vector:
    n = len(U)
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        s = sum(U[i][k] * x[k] for k in range(i + 1, n))
        x[i] = (b[i] - s) / U[i][i]
    return x


def solve_pos_def(A: Matrix, b: Vector) -> Vector:
    L = cholesky(A)
    y = solve_lower_triangular(L, b)
    x = solve_upper_triangular(transpose(L), y)
    return x


def invert_pos_def(A: Matrix) -> Matrix:
    n = len(A)
    inv = zeros(n, n)
    L = cholesky(A)
    Lt = transpose(L)
    for i in range(n):
        e = [0.0] * n
        e[i] = 1.0
        y = solve_lower_triangular(L, e)
        x = solve_upper_triangular(Lt, y)
        for j in range(n):
            inv[j][i] = x[j]
    return inv


def scale_vector(v: Vector, scalar: float) -> Vector:
    return [scalar * x for x in v]


def add_vectors(a: Vector, b: Vector) -> Vector:
    return [x + y for x, y in zip(a, b)]


def quantiles(values: List[Vector], probs: Iterable[float]) -> List[Vector]:
    if not values:
        return []
    sorted_values = [sorted(col) for col in zip(*values)]
    n = len(values)
    result: List[Vector] = []
    for p in probs:
        if not 0.0 <= p <= 1.0:
            raise ValueError("Probabilities must lie in [0, 1].")
        index = p * (n - 1)
        lower = int(math.floor(index))
        upper = int(math.ceil(index))
        weight = index - lower
        quantile = [
            (1 - weight) * sorted_values[j][lower] + weight * sorted_values[j][upper]
            for j in range(len(sorted_values))
        ]
        result.append(quantile)
    return result


def vector_quantiles(values: List[float], probs: Iterable[float]) -> List[float]:
    if not values:
        return []
    sorted_vals = sorted(values)
    n = len(values)
    result = []
    for p in probs:
        if not 0.0 <= p <= 1.0:
            raise ValueError("Probabilities must lie in [0, 1].")
        index = p * (n - 1)
        lower = int(math.floor(index))
        upper = int(math.ceil(index))
        weight = index - lower
        result.append((1 - weight) * sorted_vals[lower] + weight * sorted_vals[upper])
    return result


def vector_mean(values: List[Vector]) -> Vector:
    if not values:
        return []
    length = len(values[0])
    totals = [0.0] * length
    for value in values:
        for i, x in enumerate(value):
            totals[i] += x
    return [total / len(values) for total in totals]


def vector_variance(values: List[float]) -> float:
    mean = sum(values) / len(values)
    return sum((x - mean) ** 2 for x in values) / len(values)


## `src/diabetes_linear_model/model_selection.py`

In [ ]:
"""Bayesian model selection via Gibbs sampling for the g-prior."""
from __future__ import annotations

import math
import random
from dataclasses import dataclass
from typing import List

from .gprior import analytic_posterior_mean, analytic_ssr_g
from .linalg import Matrix, Vector, cholesky, matmul, quantiles, transpose, vector_mean, vector_quantiles


@dataclass
class ModelSelectionResult:
    """Posterior summaries returned by :class:`GPriorModelSelector`."""

    inclusion_probabilities: List[float]
    beta_mean: Vector
    beta_ci: List[Vector]
    sigma2_mean: float
    sigma2_ci: tuple[float, float]
    beta_samples: List[Vector]
    sigma2_samples: List[float]
    z_samples: List[List[int]]


def _subset_design_matrix(X: Matrix, z: List[int]) -> Matrix:
    columns = [0] + [idx + 1 for idx, value in enumerate(z) if value == 1]
    subset = []
    for row in X:
        subset.append([row[idx] for idx in columns])
    return subset


def _log_marginal_likelihood(X: Matrix, y: Vector, g: float, nu0: float, sigma0_sq: float) -> float:
    n = len(y)
    p = len(X[0])
    ssr_g = analytic_ssr_g(X, y, g)
    return (
        -0.5 * n * math.log(math.pi)
        + math.lgamma(0.5 * (nu0 + n))
        - math.lgamma(0.5 * nu0)
        - 0.5 * p * math.log(1.0 + g)
        + 0.5 * nu0 * math.log(nu0 * sigma0_sq)
        - 0.5 * (nu0 + n) * math.log(nu0 * sigma0_sq + ssr_g)
    )


class GPriorModelSelector:
    """Gibbs sampler for Bayesian variable selection with a g-prior."""

    def __init__(
        self,
        X: Matrix,
        y: Vector,
        g: float,
        nu0: float,
        sigma0_sq: float,
        inclusion_prob: float = 0.5,
        rng: random.Random | None = None,
    ) -> None:
        if len(X) != len(y):
            raise ValueError("X and y must have the same number of observations.")
        if not 0.0 < inclusion_prob < 1.0:
            raise ValueError("The prior inclusion probability must lie in (0, 1).")
        self.X = X
        self.y = y
        self.g = g
        self.nu0 = nu0
        self.sigma0_sq = sigma0_sq
        self.inclusion_prob = inclusion_prob
        self.rng = rng or random.Random()
        self.num_predictors = len(X[0]) - 1
        if self.num_predictors < 0:
            raise ValueError("Design matrix must contain at least an intercept column.")

    def _conditional_inclusion_probability(self, z: List[int], j: int) -> float:
        z_with = z.copy()
        z_with[j] = 1
        z_without = z.copy()
        z_without[j] = 0

        X_with = _subset_design_matrix(self.X, z_with)
        X_without = _subset_design_matrix(self.X, z_without)

        log_ml_with = _log_marginal_likelihood(X_with, self.y, self.g, self.nu0, self.sigma0_sq)
        log_ml_without = _log_marginal_likelihood(X_without, self.y, self.g, self.nu0, self.sigma0_sq)

        log_prior_odds = math.log(self.inclusion_prob) - math.log(1.0 - self.inclusion_prob)
        log_odds = log_prior_odds + (log_ml_with - log_ml_without)
        return 1.0 / (1.0 + math.exp(-log_odds))

    def run(self, num_samples: int, burn_in: int = 1000, thin: int = 1) -> ModelSelectionResult:
        if num_samples <= 0:
            raise ValueError("num_samples must be positive.")
        if burn_in < 0:
            raise ValueError("burn_in cannot be negative.")
        if thin <= 0:
            raise ValueError("thin must be positive.")

        total_iterations = burn_in + num_samples * thin
        z = [1] * self.num_predictors

        beta_samples: List[Vector] = []
        sigma2_samples: List[float] = []
        z_samples: List[List[int]] = []

        lower = 0.025
        upper = 0.975

        for iteration in range(total_iterations):
            for j in range(self.num_predictors):
                prob = self._conditional_inclusion_probability(z, j)
                z[j] = 1 if self.rng.random() < prob else 0

            X_current = _subset_design_matrix(self.X, z)
            beta_mean = analytic_posterior_mean(X_current, self.y, self.g)
            ssr_g = analytic_ssr_g(X_current, self.y, self.g)

            shape = 0.5 * (self.nu0 + len(self.X))
            scale = 0.5 * (self.nu0 * self.sigma0_sq + ssr_g)
            sigma2 = 1.0 / self.rng.gammavariate(shape, 1.0 / scale)

            # Sample beta conditional on sigma2.
            Xt = transpose(X_current)
            XtX = matmul(Xt, X_current)
            chol = cholesky(XtX)
            std_normals = [self.rng.gauss(0.0, 1.0) for _ in beta_mean]
            Lt = transpose(chol)
            v = [0.0] * len(beta_mean)
            for i in range(len(beta_mean) - 1, -1, -1):
                s = sum(Lt[i][k] * v[k] for k in range(i + 1, len(beta_mean)))
                v[i] = (std_normals[i] - s) / Lt[i][i]
            scale_factor = math.sqrt(self.g / (self.g + 1.0) * sigma2)
            beta_draw = [beta_mean[i] + scale_factor * v[i] for i in range(len(beta_mean))]

            full_beta = [0.0] * (self.num_predictors + 1)
            full_beta[0] = beta_draw[0]
            included_indices = [idx for idx, value in enumerate(z) if value == 1]
            for position, idx in enumerate(included_indices, start=1):
                full_beta[idx + 1] = beta_draw[position]

            if iteration >= burn_in and (iteration - burn_in) % thin == 0:
                beta_samples.append(full_beta)
                sigma2_samples.append(sigma2)
                z_samples.append(z.copy())

        inclusion_probabilities = [sum(column) / len(z_samples) for column in zip(*z_samples)] if z_samples else [0.0] * self.num_predictors
        beta_ci = quantiles(beta_samples, [lower, upper])
        sigma2_ci = tuple(vector_quantiles(sigma2_samples, [lower, upper]))

        return ModelSelectionResult(
            inclusion_probabilities=inclusion_probabilities,
            beta_mean=vector_mean(beta_samples),
            beta_ci=beta_ci,
            sigma2_mean=sum(sigma2_samples) / len(sigma2_samples),
            sigma2_ci=sigma2_ci,
            beta_samples=beta_samples,
            sigma2_samples=sigma2_samples,
            z_samples=z_samples,
        )


## `tests/conftest.py`

In [ ]:
import sys
from pathlib import Path

# Ensure the src directory is on the Python path for tests.
ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


## `tests/test_gprior.py`

In [ ]:
import random

from diabetes_linear_model.gprior import analytic_posterior_mean, fit_g_prior_model
from diabetes_linear_model.linalg import vecdot


def test_posterior_mean_matches_closed_form():
    rng = random.Random(123)
    n, p = 40, 3
    X = [[rng.gauss(0.0, 1.0) for _ in range(p)] for _ in range(n)]
    beta_true = [1.5, -2.0, 0.5]
    y = [vecdot(row, beta_true) + rng.gauss(0.0, 0.5) for row in X]

    g = float(n)
    nu0 = 2.0
    sigma0_sq = 1.0

    posterior = fit_g_prior_model(X, y, g=g, nu0=nu0, sigma0_sq=sigma0_sq, num_samples=2000, rng=rng)
    expected_mean = analytic_posterior_mean(X, y, g)

    for est, expected in zip(posterior.beta_mean, expected_mean):
        assert abs(est - expected) < 0.3
    assert len(posterior.beta_ci) == 2
    assert posterior.sigma2_ci[0] > 0


## `tests/test_model_selection.py`

In [ ]:
import random

from diabetes_linear_model.model_selection import GPriorModelSelector
from diabetes_linear_model.linalg import vecdot


def test_model_selection_outputs_shapes():
    rng = random.Random(42)
    n = 60
    X_base = [[rng.gauss(0.0, 1.0) for _ in range(2)] for _ in range(n)]
    X = [[1.0] + row for row in X_base]
    beta = [0.5, 1.0, 0.0]
    y = [vecdot(row, beta) + rng.gauss(0.0, 0.3) for row in X]

    selector = GPriorModelSelector(X, y, g=float(n), nu0=2.0, sigma0_sq=1.0, rng=rng)
    result = selector.run(num_samples=200, burn_in=50, thin=2)

    assert len(result.beta_samples[0]) == len(X[0])
    assert len(result.z_samples[0]) == len(X[0]) - 1
    assert all(0.0 <= prob <= 1.0 for prob in result.inclusion_probabilities)
    assert len(result.beta_ci) == 2
    assert result.sigma2_ci[0] > 0
